# GRAM-Inspired Recurrent-Depth Qwen

Colab handoff notebook for the staged build:

1. Run unit tests.
2. Pass the Phase 0 identity gate.
3. Smoke-test deterministic halting.
4. Smoke-test stochastic latent trajectories.

Use `Qwen/Qwen2.5-0.5B-Instruct` first. Do not scale model size until the gates pass.

## 0. Runtime

In Colab Pro+, set `Runtime > Change runtime type` to:

- Runtime type: `Python 3`
- Hardware accelerator: `H100 GPU` if available, then `A100 GPU`, then `L4 GPU`
- High-RAM: `On`
- Runtime version: `Latest (recommended)`

Skip TPUs for this repo; the code is PyTorch/Hugging Face CUDA oriented.

In [ ]:
import os, sys, platform, subprocess, psutil
from pathlib import Path

print('python', sys.version)
try:
    import torch
    print('torch', torch.__version__)
    print('cuda available', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('gpu', torch.cuda.get_device_name(0))
except Exception as exc:
    print('torch import failed before install:', exc)

!nvidia-smi || true
print('ram_gb', psutil.virtual_memory().total / 1e9)

## 1. Install Dependencies

In [ ]:
!pip -q install -U "transformers>=4.44" accelerate pyyaml pytest sentencepiece safetensors datasets

## 2. Load Project Code

Either paste a GitHub repo URL into `REPO_URL`, or leave it blank and upload a zip of this project directory when prompted.

In [ ]:
import os, sys, shutil, zipfile
from pathlib import Path

PROJECT_ROOT = Path('/content/gram-recurrent-qwen')
REPO_URL = ''  # Optional: paste a GitHub URL here. Blank means upload a zip.

if PROJECT_ROOT.exists():
    shutil.rmtree(PROJECT_ROOT)

def repair_windows_zip_paths(root):
    # Some Windows-created zips extract on Linux as files named `tests\\x.py`.
    # Turn those literal backslashes back into real directories.
    for path in list(root.rglob('*')):
        rel = str(path.relative_to(root))
        if '\\' not in rel:
            continue
        target = root.joinpath(*rel.split('\\'))
        target.parent.mkdir(parents=True, exist_ok=True)
        if target.exists():
            path.unlink()
        else:
            path.rename(target)

if REPO_URL.strip():
    !git clone {REPO_URL} {PROJECT_ROOT}
else:
    from google.colab import files
    uploaded = files.upload()
    zip_names = [name for name in uploaded if name.lower().endswith('.zip')]
    assert zip_names, 'Upload a .zip of the project directory.'
    with zipfile.ZipFile(zip_names[0]) as zf:
        zf.extractall(PROJECT_ROOT)
    repair_windows_zip_paths(PROJECT_ROOT)
    children = [p for p in PROJECT_ROOT.iterdir() if p.is_dir()]
    if not (PROJECT_ROOT / 'models').exists() and len(children) == 1 and (children[0] / 'models').exists():
        nested = children[0]
        temp = Path('/content/_gram_recurrent_qwen_nested')
        if temp.exists():
            shutil.rmtree(temp)
        nested.rename(temp)
        shutil.rmtree(PROJECT_ROOT)
        temp.rename(PROJECT_ROOT)

def looks_like_project(path):
    return (path / 'models').is_dir() and (path / 'tests').is_dir() and (path / 'eval').is_dir()

if not looks_like_project(PROJECT_ROOT):
    candidates = [p.parent for p in PROJECT_ROOT.rglob('test_bridge.py') if p.parent.name == 'tests']
    candidates = [p for p in candidates if looks_like_project(p)]
    assert candidates, f'Could not locate project root under {PROJECT_ROOT}. Check the uploaded zip contents.'
    PROJECT_ROOT = candidates[0]

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
print('project root:', PROJECT_ROOT)
!pwd
!find . -maxdepth 2 -type f | sort | head -80

## 3. Unit Tests

In [ ]:
import os, sys
from pathlib import Path

def repair_windows_zip_paths(root):
    for path in list(root.rglob('*')):
        rel = str(path.relative_to(root))
        if '\\' not in rel:
            continue
        target = root.joinpath(*rel.split('\\'))
        target.parent.mkdir(parents=True, exist_ok=True)
        if target.exists():
            path.unlink()
        else:
            path.rename(target)

root = Path.cwd()
repair_windows_zip_paths(root)
if not (root / 'tests' / 'test_bridge.py').exists():
    candidates = [p.parent.parent for p in Path('/content').rglob('tests/test_bridge.py')]
    assert candidates, 'Could not find tests/test_bridge.py under /content. Re-run the Load Project Code cell.'
    root = candidates[0]
    os.chdir(root)
    sys.path.insert(0, str(root))

os.environ['PYTHONPATH'] = str(Path.cwd()) + ':' + os.environ.get('PYTHONPATH', '')
print('running tests from', Path.cwd())
print('PYTHONPATH=', os.environ['PYTHONPATH'])
!ls tests
!python -m pytest -q tests

## 4. Phase 0 Identity Gate

This must pass before any training. For 0.5B, `6,18` is the intended split.

In [ ]:
import torch

MODEL_NAME = 'Qwen/Qwen2.5-0.5B-Instruct'
SPLIT = '6,18'
DTYPE = 'float16' if torch.cuda.is_available() else 'float32'
IDENTITY_DTYPE = 'float32'
IDENTITY_ATTN = 'eager'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(MODEL_NAME, SPLIT, IDENTITY_DTYPE, IDENTITY_ATTN, DEVICE)
!python eval/eval_identity.py --model_name {MODEL_NAME} --split {SPLIT} --dtype {IDENTITY_DTYPE} --attn_implementation {IDENTITY_ATTN} --device {DEVICE} --threshold 1e-3 | tee identity_gate.log
!grep -E "max_abs_diff|mean_abs_diff|PASS|FAIL" identity_gate.log

## 5. Deterministic Halting Telemetry

In [ ]:
!python eval/eval_halting.py --model_name {MODEL_NAME} --split {SPLIT} --max_loops 4 --dtype {DTYPE} --device {DEVICE}

## Optional: Prepare a Non-Toy Reasoning Dataset

Run this after the smoke harness works. It converts a Hugging Face reasoning dataset into this repo's `prompt` / `completion` / `cot_tokens` JSONL format.

In [ ]:
%cd /content/gram-recurrent-qwen

!python training/prepare_hf_reasoning_jsonl.py \
  --dataset_id lordx64/reasoning-distill-opus-4-7-max-sft \
  --tokenizer_name Qwen/Qwen2.5-0.5B-Instruct \
  --output_jsonl data/opus47_train.jsonl \
  --val_jsonl data/opus47_val.jsonl \
  --limit 1000 \
  --max_total_tokens 2048

!wc -l data/opus47_train.jsonl data/opus47_val.jsonl
!head -1 data/opus47_train.jsonl

## 6. Create Tiny Smoke Dataset

This is only to verify that the training loop moves, saves checkpoints, and reports telemetry. It is not a meaningful training run.

In [ ]:
from pathlib import Path

Path('data').mkdir(exist_ok=True)
Path('data/smoke_train.jsonl').write_text(
    '\n'.join([
        '{"prompt":"Solve: 2 + 3 = ","completion":"5","cot_tokens":16}',
        '{"prompt":"Find one valid 4-queens placement: ","completion":"[2, 4, 1, 3]","cot_tokens":64}',
        '{"prompt":"If x + 2 = 5, x = ","completion":"3","cot_tokens":24}'
    ]) + '\n',
    encoding='utf-8',
)
!cat data/smoke_train.jsonl

## 7. Colab Smoke Configs

In [ ]:
import yaml
from pathlib import Path

def make_smoke_config(src, dst, phase):
    cfg = yaml.safe_load(Path(src).read_text())
    cfg.update({
        'model_name': MODEL_NAME,
        'dtype': DTYPE,
        'layer_split': SPLIT,
        'max_length': 256,
        'max_loops': 2,
        'batch_size': 1,
        'max_steps': 2,
        'log_every': 1,
        'output_dir': f'outputs/colab_{phase}_smoke',
    })
    if phase == 'phase2':
        cfg['num_trajectories'] = 2
    Path(dst).write_text(yaml.safe_dump(cfg, sort_keys=False), encoding='utf-8')

make_smoke_config('config/qwen_0_5b_phase1.yaml', 'config/colab_phase1_smoke.yaml', 'phase1')
make_smoke_config('config/qwen_0_5b_phase2.yaml', 'config/colab_phase2_smoke.yaml', 'phase2')
!cat config/colab_phase1_smoke.yaml
!cat config/colab_phase2_smoke.yaml

## 8. Phase 1 Smoke Training

In [ ]:
!python training/train_phase1_ponder.py --config config/colab_phase1_smoke.yaml --train_jsonl data/smoke_train.jsonl --device {DEVICE}

## 9. Phase 2 Stochastic Trajectory Smoke Training

In [ ]:
!python training/train_phase2_stochastic.py --config config/colab_phase2_smoke.yaml --train_jsonl data/smoke_train.jsonl --device {DEVICE}

## 10. Trajectory Telemetry

In [ ]:
!python eval/eval_trajectories.py --model_name {MODEL_NAME} --split {SPLIT} --max_loops 2 --num_trajectories 2 --dtype {DTYPE} --device {DEVICE}

## 11. Save Outputs to Drive

In [ ]:
SAVE_TO_DRIVE = False

if SAVE_TO_DRIVE:
    from google.colab import drive
    import shutil
    drive.mount('/content/drive')
    target = Path('/content/drive/MyDrive/gram-recurrent-qwen-outputs')
    target.mkdir(parents=True, exist_ok=True)
    if Path('outputs').exists():
        shutil.copytree('outputs', target / 'outputs', dirs_exist_ok=True)
    print('saved to', target)
else:
    !find outputs -maxdepth 3 -type f 2>/dev/null || true